# RAG System - Indexing

Setting up the Retrieval-Augmented Generation (RAG) system with vector indexing.

In [1]:
import pdfplumber
import json
from pathlib import Path
import sys
import unicodedata
    

In [2]:
from pathlib import Path
import re
import pdfplumber

RAW_PDF_DIR = Path("../data/raw/policy_dirs")
OUT_TEXT_DIR = Path("../data/processed/hotel_policy_text")

def clean_text(t: str) -> str:
    t = t.replace("\xa0", " ")
    t = t.replace("\ufb01", "fi").replace("\ufb02", "fl")  # ligatures
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = re.sub(r"\baisons\s+preuve\b", "faisons preuve", t, flags=re.IGNORECASE)
    return t.strip()

pdf_files = sorted(RAW_PDF_DIR.glob("*.pdf"))
print("PDFs found:", len(pdf_files))

docs = []
OUT_TEXT_DIR.mkdir(parents=True, exist_ok=True)

for pdf_path in pdf_files:
    merged_text = []

    with pdfplumber.open(str(pdf_path)) as pdf:
        for page_idx, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            text = clean_text(text)

            if text:
                docs.append({
                    "text": text,
                    "source": pdf_path.name,
                    "page": page_idx
                })
                merged_text.append(text)

    if merged_text:
        out_file = OUT_TEXT_DIR / f"{pdf_path.stem}.txt"
        out_file.write_text("\n\n".join(merged_text), encoding="utf-8")

print("Pages extracted:", len(docs))
if docs:
    print("Sample:", docs[0]["source"], "p", docs[0]["page"])
    print(docs[0]["text"][:300])

PDFs found: 6
Pages extracted: 44
Sample: Convention collective - services alimentaires.pdf p 1
CONVENTION COLLECTIVE
Entre
L’HÔTEL DE LA PROMENADE
(ci-après appelé l’Employeur)
Et
LE SYNDICAT CANADIEN DES
EMPLOYÉES ET EMPLOYÉS DE LA RESTAURATION
SECTION LOCALE 155
(ci-après appelé le Syndicat)
1er janvier 2066 au 31 décembre 2069


In [3]:
d = docs[41]
print(d["source"], d["page"])
print("aisons preuve" in d["text"].lower())
print("faisons preuve" in d["text"].lower())
print(d["text"][0:250])

Principes de vie internes.pdf 2
True
True
PRINCIPES DE VIE INTERNES
3. Engagement envers l’excellence
Nous visons à atteindre l'excellence dans toutes nos tâches et responsabilités. Nous nous efforçons
de fournir un travail de haute qualité dans toutes nos actions et nous nous engageons à fo


In [4]:
# Add project root to Python path
sys.path.append(str(Path("..").resolve()))
from src.rag.chunking import chunk_docs


chunked = chunk_docs(docs, chunk_size=500, overlap=100)
print("Chunks created:", len(chunked))

out_path = Path("../data/processed/chunks/hotel_chunks.jsonl")
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w", encoding="utf-8") as f:
    for c in chunked:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print("Saved:", out_path)

C:\Users\camar\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chunks created: 180
Saved: ..\data\processed\chunks\hotel_chunks.jsonl


In [5]:
from src.rag.vector_store import build_and_save_faiss_index

CHUNKS_PATH = "../data/processed/chunks/hotel_chunks.jsonl"
INDEX_DIR = "../data/processed/index/faiss_hotel"

n_vecs, dim = build_and_save_faiss_index(CHUNKS_PATH, INDEX_DIR)

print("FAISS index saved to:", INDEX_DIR)
print("Vectors:", n_vecs, "| Dim:", dim)

Batches: 100%|██████████| 3/3 [00:06<00:00,  2.15s/it]

FAISS index saved to: ../data/processed/index/faiss_hotel
Vectors: 180 | Dim: 384
